# 🕸️ Graphs & Trees — Everything is Connected

Graphs and trees appear in ~25% of contest problems. Many problems that don't
look like graphs ARE graphs in disguise (grids, dependencies, transformations).

**Recognition Triggers:**
- "connected components" / "islands" → BFS/DFS/Union-Find
- "shortest path" → BFS (unweighted) or Dijkstra (weighted)
- "prerequisites" / "ordering" → Topological Sort
- "grid with obstacles" → Grid = graph, cells = nodes
- "tree depth / ancestor / path" → DFS on tree

In [ ]:
from collections import defaultdict, deque
from heapq import heappush, heappop

---
## 📊 Graph Representations

In [ ]:
# Adjacency List (most common in contests)
graph = defaultdict(list)
edges = [(0,1), (0,2), (1,3), (2,3), (3,4)]
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)  # Undirected
print("Adjacency List:", dict(graph))

# Adjacency Matrix (for dense graphs)
n = 5
matrix = [[0]*n for _ in range(n)]
for u, v in edges:
    matrix[u][v] = matrix[v][u] = 1
print("Adjacency Matrix:")
for row in matrix: print(" ", row)

---
## 📋 BFS and DFS Templates

In [ ]:
# BFS — Level-order, shortest path in unweighted graphs
def bfs(graph, start):
    visited = {start}
    queue = deque([start])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

# DFS — Recursive
def dfs_recursive(graph, node, visited=None):
    if visited is None: visited = set()
    visited.add(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited)
    return visited

# DFS — Iterative (avoids recursion limit)
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited: continue
        visited.add(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)
    return visited

print("BFS from 0:", bfs(graph, 0))
print("DFS from 0:", dfs_iterative(graph, 0))

---
## 🔥 Problem: Number of Islands

In [ ]:
def num_islands(grid):
    """Count connected components of '1's in a grid. Time: O(m*n), Space: O(m*n)"""
    if not grid: return 0
    rows, cols = len(grid), len(grid[0])
    count = 0
    
    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '0'  # Mark visited
        dfs(r+1, c); dfs(r-1, c); dfs(r, c+1); dfs(r, c-1)
    
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                dfs(r, c)
    return count

grid = [
    ['1','1','0','0','0'],
    ['1','1','0','0','0'],
    ['0','0','1','0','0'],
    ['0','0','0','1','1']
]
print("Number of Islands:", num_islands(grid))  # 3

---
## 🔥 Problem: Course Schedule (Topological Sort)

In [ ]:
def can_finish(num_courses, prerequisites):
    """Can you finish all courses? (No cycles in prerequisite graph)
    Uses Kahn's algorithm (BFS topological sort).
    Time: O(V + E)"""
    graph = defaultdict(list)
    in_degree = [0] * num_courses
    
    for course, prereq in prerequisites:
        graph[prereq].append(course)
        in_degree[course] += 1
    
    queue = deque([i for i in range(num_courses) if in_degree[i] == 0])
    count = 0
    
    while queue:
        node = queue.popleft()
        count += 1
        for neighbor in graph[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    return count == num_courses

print("Course Schedule:")
print("  2 courses, [[1,0]]:", can_finish(2, [[1,0]]))           # True
print("  2 courses, [[1,0],[0,1]]:", can_finish(2, [[1,0],[0,1]]))  # False (cycle)

---
## 🔥 Dijkstra's Shortest Path

In [ ]:
def dijkstra(graph, start, n):
    """Shortest path from start to all nodes. graph[u] = [(v, weight), ...]
    Time: O((V+E) log V)"""
    dist = [float('inf')] * n
    dist[start] = 0
    heap = [(0, start)]
    
    while heap:
        d, u = heappop(heap)
        if d > dist[u]: continue
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heappush(heap, (dist[v], v))
    return dist

g = defaultdict(list)
g[0] = [(1, 4), (2, 1)]
g[1] = [(3, 1)]
g[2] = [(1, 2), (3, 5)]
print("Dijkstra from 0:", dijkstra(g, 0, 4))  # [0, 3, 1, 4]

---
## 🌳 Binary Tree Problems

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Build sample tree:     1
#                       / \
#                      2   3
#                     / \   \
#                    4   5   6
root = TreeNode(1,
    TreeNode(2, TreeNode(4), TreeNode(5)),
    TreeNode(3, None, TreeNode(6)))

In [ ]:
# Tree Traversals
def inorder(node):
    if not node: return []
    return inorder(node.left) + [node.val] + inorder(node.right)

def preorder(node):
    if not node: return []
    return [node.val] + preorder(node.left) + preorder(node.right)

def postorder(node):
    if not node: return []
    return postorder(node.left) + postorder(node.right) + [node.val]

def level_order(root):
    if not root: return []
    result, queue = [], deque([root])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            if node.left: queue.append(node.left)
            if node.right: queue.append(node.right)
        result.append(level)
    return result

print("Inorder:", inorder(root))       # [4, 2, 5, 1, 3, 6]
print("Preorder:", preorder(root))     # [1, 2, 4, 5, 3, 6]
print("Postorder:", postorder(root))   # [4, 5, 2, 6, 3, 1]
print("Level-order:", level_order(root))  # [[1], [2,3], [4,5,6]]

In [ ]:
# Invert Binary Tree
def invert_tree(root):
    if not root: return None
    root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root

# Max Depth
def max_depth(root):
    if not root: return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

# Diameter of Binary Tree
def diameter(root):
    result = [0]
    def height(node):
        if not node: return 0
        left = height(node.left)
        right = height(node.right)
        result[0] = max(result[0], left + right)
        return 1 + max(left, right)
    height(root)
    return result[0]

# Is Balanced?
def is_balanced(root):
    def check(node):
        if not node: return 0
        left = check(node.left)
        right = check(node.right)
        if left == -1 or right == -1 or abs(left - right) > 1:
            return -1
        return 1 + max(left, right)
    return check(root) != -1

# Same Tree
def is_same_tree(p, q):
    if not p and not q: return True
    if not p or not q: return False
    return p.val == q.val and is_same_tree(p.left, q.left) and is_same_tree(p.right, q.right)

# Lowest Common Ancestor
def lca(root, p, q):
    if not root or root == p or root == q:
        return root
    left = lca(root.left, p, q)
    right = lca(root.right, p, q)
    if left and right: return root  # p and q are on different sides
    return left or right

print(f"Max Depth: {max_depth(root)}")      # 3
print(f"Diameter: {diameter(root)}")          # 4
print(f"Is Balanced: {is_balanced(root)}")    # True

In [ ]:
# Validate BST
def is_valid_bst(root, lo=float('-inf'), hi=float('inf')):
    if not root: return True
    if root.val <= lo or root.val >= hi: return False
    return is_valid_bst(root.left, lo, root.val) and is_valid_bst(root.right, root.val, hi)

# Kth Smallest in BST (inorder traversal)
def kth_smallest(root, k):
    stack = []
    current = root
    count = 0
    while stack or current:
        while current:
            stack.append(current)
            current = current.left
        current = stack.pop()
        count += 1
        if count == k:
            return current.val
        current = current.right
    return -1

# Build BST:   4
#             / \
#            2   6
#           / \ / \
#          1  3 5  7
bst = TreeNode(4,
    TreeNode(2, TreeNode(1), TreeNode(3)),
    TreeNode(6, TreeNode(5), TreeNode(7)))

print(f"Valid BST: {is_valid_bst(bst)}")           # True
print(f"3rd smallest: {kth_smallest(bst, 3)}")     # 3

---
## 🏆 Summary

| Problem | Algorithm | Time | Space |
|---------|-----------|------|-------|
| Number of Islands | DFS/BFS on grid | O(mn) | O(mn) |
| Course Schedule | Topological Sort (BFS) | O(V+E) | O(V+E) |
| Dijkstra | Min-heap BFS | O((V+E)logV) | O(V) |
| Tree traversals | DFS (recursive) | O(n) | O(h) |
| Level order | BFS | O(n) | O(n) |
| Max depth / diameter | DFS | O(n) | O(h) |
| Validate BST | DFS with bounds | O(n) | O(h) |
| Kth smallest BST | Inorder traversal | O(h+k) | O(h) |
| LCA | DFS | O(n) | O(h) |